<a href="https://colab.research.google.com/github/AlejandraLopR/Hands_On_Prompt_Engineering_y_Sistemas_RAG/blob/main/Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente de Groq inicializado correctamente')

Cliente de Groq inicializado correctamente


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [7]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de la reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."
response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt_zero_shot}]
)

print("Zero-shot:\n", response_zero.choices[0].message.content)

Zero-shot:
 Mixto


In [11]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llego rapido y   en perfecto estado".
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio",
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente",
Sentimiento: """

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content": prompt_few_shot}]
)

print("Few-shot", response_few.choices[0].message.content)

Few-shot Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [12]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale a 80 km/h. Dos horas después, otro tren sale de la misma."
    "ciudad hacia el mismo destino a 120km/h. ¿Cuánto tiempo tarda el segundo tren en"
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)


response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)

**Paso 1 – Identificar los datos conocidos**

| Variable | Valor | Unidades |
|----------|-------|----------|
| Velocidad del primer tren \(v_1\) | 80 | km/h |
| Velocidad del segundo tren \(v_2\) | 120 | km/h |
| Tiempo de diferencia de salida | 2 | h |

**Paso 2 – Calcular la ventaja (distancia adelantada) del primer tren**

El primer tren viaja durante 2 horas antes de que salga el segundo tren, por lo que su ventaja es

\[
\text{Ventaja} = v_1 \times \text{tiempo de diferencia}
= 80 \text{ km/h} \times 2 \text{ h}
= 160 \text{ km}.
\]

**Paso 3 – Determinar la velocidad relativa (diferencia de velocidades)**

El segundo tren se mueve más rápido que el primero. La velocidad con la que reduce la distancia entre ambos trenes es la diferencia de sus velocidades:

\[
v_{\text{rel}} = v_2 - v_1
= 120 \text{ km/h} - 80 \text{ km/h}
= 40 \text{ km/h}.
\]

**Paso 4 – Calcular el tiempo necesario para que el segundo tren alcance al primero**

El tiempo \(t\) que tardará en cerrar la brecha d

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [13]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, pero no dispongo de información sobre ese evento.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [14]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [15]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [16]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [17]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

No, los productos en oferta no se pueden devolver, solo cambiar de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [18]:
# Leer API key, instalar e importar librerías
!pip install groq sentence-transformers --quiet

from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import numpy as np

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente de Groq inicializado correctamente')

Cliente de Groq inicializado correctamente


In [20]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Regular los apoyos que el Gobierno Federal está obligado a otorgar para impulsar, fortalecer,desarrollar y consolidar la investigación científica, "
    "el desarrollo tecnológico y la innovación en general en el país",
    "Determinar los instrumentos mediante los cuales el Gobierno Federal cumplirá con la obligación de apoyar la investigación científica,"
    " el desarrollo tecnológico y la innovación",
    "Establecer los mecanismos de coordinación de acciones entre las dependencias y entidades de la Administración Pública Federal y otras "
    "instituciones que intervienen en la definición de políticas y programas en materia de desarrollo científico, tecnológico e innovación, "
    "o que lleven a cabo directamente actividades de este tipo"
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [32]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Qué busca regular la ley de ciencia y datos?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Determinar los instrumentos mediante los cuales el Gobierno Federal cumplirá con la obligación de apoyar la investigación científica, el desarrollo tecnológico y la innovación


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [35]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

prompt_rag = f"""Responde la pregunta SOLO con la siguiente política de la ley para regular el apoyo a la investigación científica. Si la política no cubre la pregunta, dilo claramente.

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(respuesta_sin_rag.choices[0].message.content)

La política de apoyo a la investigación científica establece que el Estado promoverá y financiará la investigación básica y aplicada, fomentará la formación de recursos humanos, y brindará incentivos fiscales y becas a investigadores.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [41]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag

prompt_rag = f"""Responde la pregunta SOLO con la siguiente política de la ley para regular el apoyo a la investigación científica. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(respuesta_con_rag.choices[0].message.content)

Regula los instrumentos mediante los cuales el Gobierno Federal apoyará la investigación científica, el desarrollo tecnológico y la innovación.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [43]:
# Mostrar ambas respuestas para comparar
print("==================================================================")
print("             COMPARACIÓN DE RESPUESTAS: RAG VS SIN RAG            ")
print("==================================================================")

print(f"\nPREGUNTA REALIZADA:\n{prompt_rag}\n")

print("-" * 66)
print("1. RESPUESTA CON RAG (Usando el fragmento de la política):")
print("-" * 66)
print(respuesta_con_rag.choices[0].message.content)

print("\n" + "-" * 66)
print("2. RESPUESTA SIN RAG (Sin contexto previo):")
print("-" * 66)
print(respuesta_sin_rag.choices[0].message.content)

print("\n==================================================================")
print("                           CONCLUSIÓN                             ")
print("==================================================================")
print("¿Cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa?\n")
print("La respuesta CON RAG fue superior y evitó de forma efectiva la alucinación.")
print("• Con RAG: El modelo se apegó estrictamente a la política del documento provisto. Su respuesta fue mas asertada y apegada a la información proporcionada.")

             COMPARACIÓN DE RESPUESTAS: RAG VS SIN RAG            

PREGUNTA REALIZADA:
Responde la pregunta SOLO con la siguiente política de la ley para regular el apoyo a la investigación científica. Si la política no cubre la pregunta, dilo claramente.

Política: Determinar los instrumentos mediante los cuales el Gobierno Federal cumplirá con la obligación de apoyar la investigación científica, el desarrollo tecnológico y la innovación

Pregunta: ¿Qué busca regular la ley de ciencia y datos?

Respuesta muy breve y corta.

------------------------------------------------------------------
1. RESPUESTA CON RAG (Usando el fragmento de la política):
------------------------------------------------------------------
Regula los instrumentos mediante los cuales el Gobierno Federal apoyará la investigación científica, el desarrollo tecnológico y la innovación.

------------------------------------------------------------------
2. RESPUESTA SIN RAG (Sin contexto previo):
-------------------